<a href="https://colab.research.google.com/github/anagha0601/Research-paper-vetting-system/blob/main/ResearchPaperVetting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Scientific Paper Quality Assessment Pipeline

This notebook evaluates scientific papers across five quality dimensions:

1. **Citation Worthiness** — Does the paper support its claims with citations?
2. **Cognitive Load** — How readable, jargon-heavy, and structurally complex is the paper?
3. **Argument Quality** — Are the conclusions justified by the evidence?
4. **Peer Review Readiness** — What concerns might a reviewer raise?
5. **Headline Accuracy** — Does the title accurately reflect the findings?

Each module produces its own assessment, and a final unified model combines all five into a single paper quality score.

## 0. Setup

### 0.1 Install Dependencies

In [ ]:
!pip install nlpaug

### 0.2 Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import re
import unicodedata
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, confusion_matrix,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)
from scipy.sparse import hstack, csr_matrix
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from nlpaug.augmenter.word import SynonymAug

nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

warnings.filterwarnings("ignore")

### 0.3 Global Configuration

In [ ]:
# Paths
# DATA_PATH = "s2orc_subset (6).csv"
DATA_PATH = "/content/s2orc_subset (6).csv"

# Reproducibility
RANDOM_STATE = 42

# Citation module
CITATION_MODEL_NAME = "allenai/scibert_scivocab_uncased"
CITATION_MAX_LEN    = 128
CITATION_BATCH_SIZE = 16
CITATION_EPOCHS     = 3
CITATION_LR         = 3e-5
CITATION_N_SPLITS   = 5
CITATION_OUTPUT_DIR = "./scibert_citation_kfold"
CITATION_TARGET_NAMES = ["No Citation", "Citation Needed"]

### 0.4 Shared Helper Functions

In [ ]:
def preprocess_text(text):
    """Normalize and clean text for downstream NLP tasks."""
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = str(text)
    text = unicodedata.normalize("NFKC", text)                # preserve scientific symbols
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)  # remove control chars
    text = text.lower()
    text = re.sub(r'[“”„]', '"', text)         # normalize quotes
    text = re.sub(r"[\u2013\u2014]", "-", text)              # normalize dashes
    text = re.sub(r"\.{2,}", "...", text)                     # normalize ellipsis
    text = re.sub(r"\s+", " ", text).strip()
    return text


def contains_any(text, keywords):
    """Return True if any keyword appears in the text."""
    if pd.isna(text):
        return False
    text = str(text).lower()
    return any(kw in text for kw in keywords)


def count_keywords(text, vocab):
    """Count occurrences of hand-crafted keyword cues in text."""
    text = str(text)
    return sum(text.count(w) for w in vocab)

## 1. Data Loading and Cleaning

### 1.1 Load Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.head())

### 1.2 Handle Missing Values and Duplicates

In [ ]:
print("Initial dataset shape:", df.shape)

key_text_cols = ["title","abstract","results"]
key_lablel_cols = [
    "exaggeration_score",
    "headline_verdict",
    "citation_needed",
    "citation_score",
    "overclaim",
    "argument_strength",
    "logical_consistency",
    "readability_score",
    "jargon_density",
    "structural_complexity",
    "strength_flag",
    "weakness_flag",
    "missing_element",
    "reviewer_concern"
]
print("Missing values before cleaning")
print(df[key_text_cols + key_lablel_cols].isnull().sum())
df["title_issues"] = df["title_issues"].fillna("None")
df["abstract_issues"] = df["abstract_issues"].fillna("None")
rows_before = len(df)

df=df.dropna(subset= key_text_cols + key_lablel_cols)
df = df[df['abstract'].str.strip() != '']
df = df[df['title'].str.strip() != '']
df=df.drop_duplicates()

rows_after = len(df)

print("\nDataset shape after cleaning:",df.shape)
print("Rows removed:", rows_before - rows_after)

### 1.3 Type Conversion & Normalization

In [ ]:
print(df.head())

In [ ]:
#convert boolean
bool_cols = ["citation_needed","overclaim","strength_flag","weakness_flag","missing_element","reviewer_concern"]

for col in bool_cols:
    if df[col].dtype == object:
        df[col] = df[col].str.strip().str.upper().map({"TRUE": 1, "FALSE": 0})
    else:
        df[col] = df[col].astype(int)

# Numeric score columns
score_cols = [
      "exaggeration_score",
      "citation_score",
      "argument_strength",
      "logical_consistency",
      "readability_score",
      "jargon_density",
      "structural_complexity"
  ]
for col in score_cols:
  df[col] = pd.to_numeric(df[col],errors="coerce")

for col in score_cols:
    df[col] = df[col] / 100

print(df[score_cols].describe().loc[["min", "max"]])

# headline_verdict is a categorical string - encode it
verdict_mapping = {
    "Accurate": 0,
    "Moderately Exaggerated": 1,
    "Significantly Exaggerated": 2,
    "Misleading": 3
}
df["headline_verdict_encoded"] = df["headline_verdict"].map(verdict_mapping)

# Check label values
print("Citation label values:", df["citation_needed"].unique())
print("Overclaim label values:", df["overclaim"].unique())
print("Argument strength range:", df["argument_strength"].min(), "-", df["argument_strength"].max())
print(df["argument_strength"].describe())
print("Headline verdict label values:", df["headline_verdict_encoded"].unique())

### 1.4 Text Preprocessing & Feature Engineering

In [ ]:
# Apply preprocessing to all text columns
text_cols = ["title", "abstract", "results", "title_issues", "abstract_issues"]
for col in text_cols:
    df[col] = df[col].apply(preprocess_text)

print("\nText preprocessing complete")
print(df[["title", "abstract"]].head(2))

In [ ]:
# Create issue flag columns
df["title_issues_flag"] = df["title_issues"].apply(
    lambda x: 0 if str(x).strip().lower() == "none" else 1
)
df["abstract_issues_flag"] = df["abstract_issues"].apply(
    lambda x: 0 if str(x).strip().lower() == "none" else 1
)

print("\nIssue flags created")
print(df[["title_issues_flag", "abstract_issues_flag"]].value_counts())

In [ ]:
# Text length features
df["title_length"] = df["title"].apply(lambda x: len(x.split()))
df["abstract_length"] = df["abstract"].apply(lambda x: len(x.split()))
df["results_length"] = df["results"].apply(lambda x: len(x.split()))

print("\nTitle length statistics:")
print(df["title_length"].describe())
print("\nAbstract length statistics:")
print(df["abstract_length"].describe())

# Remove paper with short abstracts (< 20 words)
df = df[df["abstract_length"] >= 20]

# Remove rows with URLs/DOI in abstract
df = df[~df["abstract"].str.contains("http|doi|www", case=False,na=False)]

print(f"\nFinal dataset shape: {df.shape}")
print(df[["title","abstract_length","citation_needed","overclaim","argument_strength"]].head())

## 2. Exploratory Data Analysis

In [ ]:
print("\n" + "="*60)
print("Exploratory Data Analysis")
print("="*60 )

# Total Samples
print(f"\nTotal samples:{len(df)}")
print(f"Total columns:{df.shape[1]}")

# Class Distribution
label_cols = [
    "citation_needed",
    "overclaim",
    "argument_strength",
    "headline_verdict_encoded",
    "strength_flag",
    "weakness_flag",
    "missing_element",
    "reviewer_concern"
]
print("\nClass distribution:")
print("-"*40)
for col in label_cols:
    print(f"{col}:")
    counts = df[col].value_counts()
    total = len(df)
    print(f"\n{col}:")
    for val, count in counts.items():
         print(f"Class {val} : {count} samples ({count/total*100:.1f}%)")

# Sentence Length Statistics
print("\nText length Statistics:")
print("-"*40)
for col in ["title_length", "abstract_length", "results_length"]:
    stats = df[col].describe()
    print(f"\n{col}:")
    print(f"Mean:{stats['mean']:.1f} words")
    print(f"Min:{stats['min']:.0f}  words")
    print(f"Max:{stats['max']:.0f}  words")
    print(f"Std:{stats['std']:.1f}  words")

# Label Imbalance Check
print("\nLabel Imbalance Check:")
print("-"*40)
binary_cols = ["citation_needed", "overclaim", "strength_flag", "weakness_flag", "missing_element", "reviewer_concern"]
for col in binary_cols:
  ratio = df[col].value_counts(normalize=True)
  imbalance = abs(ratio.get(0, 0) - ratio.get(1,0))
  if imbalance > 0.3:
    status = "imbalanced"
  else:
    status = "balanced"
    print(f"{col} is balanced")
  print(f"  {col:25s} : {status} (0: {ratio.get(0,0):.2f} / 1: {ratio.get(1,0):.2f})")

In [ ]:
# Sample Examples per class
print("\nSample Examples per class:")
print("-"*40)
verdict_name={
    0:"Accurate",
    1:"Moderately Exaggerated",
    2:"Significantly Exaggerated",
    3:"Misleading"
}
for code, name in verdict_name.items():
    sample = df[df["headline_verdict_encoded"] == code]["title"].head(1).values
    if len(sample) > 0:
        print(f"\n  [{name}]")
        print(f"  → {sample[0]}")

In [ ]:
# Score Distribution Plots
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(score_cols):
    axes[i].hist(df[col], bins=10, color="steelblue", edgecolor="white")
    axes[i].set_title(col.replace("_", " ").title(), fontsize=10)
    axes[i].set_xlabel("Score (normalized)")
    axes[i].set_ylabel("Count")

axes[-1].axis("off")
plt.suptitle("Score Column Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(label_cols):
    df[col].value_counts().sort_index().plot(
        kind="bar", ax=axes[i], color="steelblue", edgecolor="white"
    )
    axes[i].set_title(col.replace("_", " ").title(), fontsize=10)
    axes[i].set_xlabel("Class")
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis="x", rotation=0)

axes[-1].axis("off")
plt.suptitle("Label Class Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Module 1 — Citation Worthiness

Predicts whether a scientific claim requires a supporting citation. Binary classification where 1 = citation needed, 0 = no citation required. Compares Logistic Regression (TF-IDF + features) vs SciBERT + numeric features using stratified K-fold cross-validation.

### 3.1 Feature Engineering

In [ ]:
# Keyword lists for numeric feature extraction
CLAIM_WORDS = [
    "demonstrate", "demonstrates", "show", "shows", "prove", "proves",
    "confirm", "confirms", "significant", "significantly", "associated",
    "association", "increase", "decrease", "reduce", "improve", "higher",
    "lower", "effective", "efficacy", "impact", "linked", "predict",
    "predicts", "cause", "causes"
]
ABSOLUTE_WORDS = [
    "all", "always", "never", "definitely", "clearly",
    "must", "cannot", "guarantee", "establishes"
]
HEDGE_WORDS = [
    "may", "might", "could", "suggest", "suggests",
    "appears", "likely", "potentially", "possibly"
]
CITATION_MARKERS = r"\[(?:\d+(?:,\s*\d+)*)\]|\([A-Za-z][A-Za-z\-\s]+,\s*\d{4}[a-z]?\)|et al\.,?\s*\d{4}"

def extract_numeric_features(texts):
    """Extract handcrafted numeric features from text for citation prediction."""
    features = []
    for text in texts:
        text = str(text)
        num_count             = len(re.findall(r"\b\d+(?:\.\d+)?%?\b", text))
        citation_marker_count = len(re.findall(CITATION_MARKERS, text))
        claim_word_count      = count_keywords(text, CLAIM_WORDS)
        absolute_word_count   = count_keywords(text, ABSOLUTE_WORDS)
        hedge_word_count      = count_keywords(text, HEDGE_WORDS)
        evidence_ratio        = citation_marker_count / (num_count + claim_word_count + 1)
        features.append([
            num_count, citation_marker_count, claim_word_count,
            absolute_word_count, hedge_word_count, evidence_ratio
        ])
    return np.array(features, dtype=np.float32)

### 3.2 Data Preparation

In [ ]:
# Primary input text
df["input_text"] = df["title"] + "[SEP]" + df["abstract"]
X = df["input_text"]
y = df["citation_needed"]

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.50, random_state=RANDOM_STATE, stratify=y_train_val
)

In [ ]:
# Input text for SciBERT (includes results)
df["citation_input_text"] = (
    df["title"] + " [SEP] " +
    df["abstract"] + " [SEP] " +
    df["results"].fillna("")
)

X_all      = df["citation_input_text"].values
y_all      = df["citation_needed"].values
issues_all = df[["title_issues_flag", "abstract_issues_flag"]].values.astype(np.float32)

### 3.3 Model Architecture

In [ ]:
# Dataset class for SciBERT
class CitationDataset(Dataset):
    def __init__(self, encodings, labels, numeric_features):
        self.encodings        = encodings
        self.labels           = list(labels)
        self.numeric_features = numeric_features

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"]           = torch.tensor(self.labels[idx], dtype=torch.long)
        item["numeric_features"] = torch.tensor(self.numeric_features[idx], dtype=torch.float)
        return item


# SciBERT + numeric features model
class SciBERTWithFeatures(nn.Module):
    def __init__(self, model_name, num_numeric_features, num_labels=2):
        super().__init__()
        self.bert       = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Sequential(
            nn.Linear(self.bert.config.hidden_size + num_numeric_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_labels)
        )

    def forward(self, input_ids, attention_mask, token_type_ids=None,
                numeric_features=None, labels=None):
        outputs    = self.bert(input_ids=input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids)
        cls_output = outputs.last_hidden_state[:, 0, :]

        combined = torch.cat([cls_output, numeric_features], dim=1) \
                   if numeric_features is not None else cls_output

        logits = self.classifier(combined)
        loss   = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None

        return SequenceClassifierOutput(loss=loss, logits=logits)

num_numric = 8

In [ ]:
# Custom Trainer with class-weighted loss
class FeatureWeightedTrainer(Trainer):
    def __init__(self, cw_tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.cw_tensor = cw_tensor

    def compute_loss(by, model, inputs, return_outputs=False, **kwargs):
        labels           = inputs.pop("labels")
        numeric_features = inputs.pop("numeric_features", None)
        outputs          = model(**inputs, numeric_features=numeric_features)
        loss             = nn.CrossEntropyLoss(weight=self.cw_tensor)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss


device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(CITATION_MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        list(texts), padding="max_length",
        truncation=True, max_length=CITATION_MAX_LEN, return_tensors="pt"
    )

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1"      : f1_score(labels, preds, average="weighted")
    }

### 3.4 K-Fold Training & Evaluation

In [ ]:
skf             = StratifiedKFold(n_splits=CITATION_N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
lr_results      = []
scibert_results = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X_all, y_all)):
    print(f"\n{'='*60}")
    print(f"  FOLD {fold+1} / {CITATION_N_SPLITS}")
    print(f"{'='*60}")

    X_train_fold = X_all[train_idx]
    y_train_fold = y_all[train_idx]
    X_test_fold  = X_all[test_idx]
    y_test_fold  = y_all[test_idx]
    issues_train = issues_all[train_idx]
    issues_test  = issues_all[test_idx]

    print(f"  Train: {len(X_train_fold)} | Test: {len(X_test_fold)}")

    # Augmentation
    aug           = SynonymAug(aug_src="wordnet")
    no_cite_texts = X_train_fold[y_train_fold == 0].tolist()
    no_cite_short = [" ".join(str(t).split()[:50]) for t in no_cite_texts]

    aug_texts = []
    for t in no_cite_short:
        try:
            result = aug.augment(t)
            aug_texts.append(result[0] if result else t)
        except Exception:
            aug_texts.append(t)

    aug_labels  = [0] * len(aug_texts)
    X_train_aug = list(X_train_fold) + aug_texts
    y_train_aug = list(y_train_fold) + aug_labels
    aug_issues  = np.zeros((len(aug_texts), 2), dtype=np.float32)

    print(f"  After aug: {len(X_train_aug)} | Test: {len(X_test_fold)}")

    # Logistic Regression
    print(f"\n  [Fold {fold+1}] Logistic Regression...")

    tfidf            = TfidfVectorizer(max_features=5000, ngram_range=(1,2),
                                       stop_words="english", sublinear_tf=True)
    X_train_tfidf    = tfidf.fit_transform(X_train_aug)
    X_test_tfidf     = tfidf.transform(X_test_fold)

    num_train        = np.hstack([extract_numeric_features(X_train_aug),
                                  np.vstack([issues_train, aug_issues])])
    num_test         = np.hstack([extract_numeric_features(X_test_fold), issues_test])

    scaler           = StandardScaler()
    num_train_sc     = scaler.fit_transform(num_train)
    num_test_sc      = scaler.transform(num_test)

    X_train_combined = hstack([X_train_tfidf, csr_matrix(num_train_sc)])
    X_test_combined  = hstack([X_test_tfidf,  csr_matrix(num_test_sc)])

    lr_model         = LogisticRegression(random_state=RANDOM_STATE, max_iter=1000,
                                          class_weight="balanced")
    lr_model.fit(X_train_combined, y_train_aug)
    lr_preds         = lr_model.predict(X_test_combined)

    lr_acc = accuracy_score(y_test_fold, lr_preds)
    lr_f1  = f1_score(y_test_fold, lr_preds, average="weighted")
    lr_results.append({"fold": fold+1, "accuracy": lr_acc, "f1": lr_f1})

    print(classification_report(y_test_fold, lr_preds, target_names=CITATION_TARGET_NAMES))
    print(f"  LR  Accuracy: {lr_acc:.4f} | F1: {lr_f1:.4f}")

    # SciBERT
    print(f"\n  [Fold {fold+1}] SciBERT...")

    num_train_sb  = np.vstack([
        np.hstack([extract_numeric_features(X_train_fold), issues_train]),
        np.hstack([extract_numeric_features(aug_texts),    aug_issues])
    ])
    num_test_sb   = np.hstack([extract_numeric_features(X_test_fold), issues_test])

    train_enc     = tokenize(X_train_aug)
    test_enc      = tokenize(X_test_fold.tolist())
    train_dataset = CitationDataset(train_enc, y_train_aug, num_train_sb)
    test_dataset  = CitationDataset(test_enc,  y_test_fold, num_test_sb)

    cw        = compute_class_weight("balanced", classes=np.array([0,1]), y=y_train_aug)
    cw_tensor = torch.tensor(cw, dtype=torch.float).to(device)

    model = SciBERTWithFeatures(CITATION_MODEL_NAME, num_numeric_features=num_numric)
    model.to(device)
    for param in model.parameters():
        param.data = param.data.contiguous()

    training_args = TrainingArguments(
        output_dir                  = f"{CITATION_OUTPUT_DIR}/fold_{fold+1}",
        optim = "adamw_torch",
        num_train_epochs            = CITATION_EPOCHS,
        per_device_train_batch_size = CITATION_BATCH_SIZE,
        per_device_eval_batch_size  = CITATION_BATCH_SIZE * 2,
        learning_rate               = CITATION_LR,
        warmup_ratio                = 0.1,
        weight_decay                 = 0.01,
        eval_strategy               = "epoch",
        save_strategy               = "no",
        load_best_model_at_end      = False,
        logging_steps               = 50,
        fp16                        = torch.cuda.is_available(),
    )

    trainer = FeatureWeightedTrainer(
        cw_tensor       = cw_tensor,
        model           = model,
        args            = training_args,
        train_dataset   = train_dataset,
        eval_dataset    = test_dataset,
        compute_metrics = compute_metrics,
    )

    trainer.train()

    output        = trainer.predict(test_dataset)
    scibert_preds = np.argmax(output.predictions, axis=-1)
    sb_acc        = accuracy_score(y_test_fold, scibert_preds)
    sb_f1         = f1_score(y_test_fold, scibert_preds, average="weighted")
    scibert_results.append({"fold": fold+1, "accuracy": sb_acc, "f1": sb_f1})

    print(classification_report(y_test_fold, scibert_preds, target_names=CITATION_TARGET_NAMES))
    print(f"  SciBERT  Accuracy: {sb_acc:.4f} | F1: {sb_f1:.4f}")

### 3.5 Results Summary

In [ ]:
lr_df      = pd.DataFrame(lr_results)
scibert_df = pd.DataFrame(scibert_results)

print(f"\n{'='*60}")
print("FINAL COMPARISON SUMMARY")
print(f"{'='*60}")

print("\nLogistic Regression (K-Fold):")
print(lr_df.to_string(index=False))
print(f"Mean Accuracy : {lr_df['accuracy'].mean():.4f} ± {lr_df['accuracy'].std():.4f}")
print(f"Mean F1 : {lr_df['f1'].mean():.4f} ± {lr_df['f1'].std():.4f}")

print("\nSciBERT + Features (K-Fold):")
print(scibert_df.to_string(index=False))
print(f"Mean Accuracy : {scibert_df['accuracy'].mean():.4f} ± {scibert_df['accuracy'].std():.4f}")
print(f" Mean F1 : {scibert_df['f1'].mean():.4f} ± {scibert_df['f1'].std():.4f}")

In [ ]:
# Best Model
lr_mean_f1      = lr_df['f1'].mean()
scibert_mean_f1 = scibert_df['f1'].mean()

print(f"\n{'='*60}")
print("  BEST MODEL")
print(f"{'='*60}")
if scibert_mean_f1 >= lr_mean_f1:
    print(f"Good model :SciBERT + Features  (F1: {scibert_mean_f1:.4f})")
    print(f"Margin: +{scibert_mean_f1 - lr_mean_f1:.4f} over Logistic Regression")
else:
    print(f" Good Model: Logistic Regression  (F1: {lr_mean_f1:.4f})")
    print(f"Margin : +{lr_mean_f1 - scibert_mean_f1:.4f} over SciBERT")

## 4. Module 2 — Cognitive Load Estimator

Predicts three continuous targets: readability score, jargon density, and structural complexity. Compares Ridge Regression vs Random Forest using TF-IDF + numeric features.

### 4.1 Feature & Target Setup


In [ ]:
# Create input text for cognitive load tasks
df["cognitive_input_text"] = df["abstract"] + " [SEP] " + df["results"]

# Features to use
feature_cols = [
    "cognitive_input_text",
    "title_length",
    "abstract_length",
    "results_length",
    "title_issues_flag",
    "abstract_issues_flag"
]

# Targets for this module
target_cols = [
    "readability_score",
    "jargon_density",
    "structural_complexity"
]

print(df[feature_cols + target_cols].head())

### 4.2 Helper Functions

In [ ]:
# Train / val / test split helper
def split_data_for_target(df, feature_cols, target_col, random_state=RANDOM_STATE):
    X = df[feature_cols].copy()
    y = df[target_col].copy()

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=0.20, random_state=random_state
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=random_state
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
# Preprocessing + model pipeline
def build_pipeline(model):
    text_features = "cognitive_input_text"
    numeric_features = [
        "title_length",
        "abstract_length",
        "results_length",
        "title_issues_flag",
        "abstract_issues_flag"
    ]

    preprocessor = ColumnTransformer(
        transformers=[
            ("text", TfidfVectorizer(max_features=5000, ngram_range=(1, 2)), text_features),
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="mean"))
            ]), numeric_features)
        ]
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    return pipeline

In [ ]:
# Evaluation helper
def evaluate_regression(model, X_train, y_train, X_val, y_val, model_name="Model"):
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))

    results = {
        "model_name": model_name,
        "train_mae": mean_absolute_error(y_train, train_pred),
        "train_rmse": train_rmse,
        "train_r2": r2_score(y_train, train_pred),
        "val_mae": mean_absolute_error(y_val, val_pred),
        "val_rmse": val_rmse,
        "val_r2": r2_score(y_val, val_pred)
    }

    return results, model

In [ ]:
# Train and compare models for one target
def run_models_for_target(df, feature_cols, target_col):
    X_train, X_val, X_test, y_train, y_val, y_test = split_data_for_target(
        df, feature_cols, target_col
    )

    models = {
        "Ridge": build_pipeline(Ridge(alpha=1.0)),
        "RandomForest": build_pipeline(
            RandomForestRegressor(
                n_estimators=200,
                random_state=RANDOM_STATE,
                n_jobs=-1
            )
        )
    }

    results_list = []
    trained_models = {}

    for name, model in models.items():
        results, trained_model = evaluate_regression(
            model, X_train, y_train, X_val, y_val, model_name=name
        )
        results["target"] = target_col
        results_list.append(results)
        trained_models[name] = trained_model

    results_df = pd.DataFrame(results_list).sort_values(by="val_rmse")
    return results_df, trained_models, X_train, X_val, X_test, y_train, y_val, y_test

### 4.3 Training & Model Selection

In [ ]:
# Run all three cognitive load tasks
all_results = {}
all_models = {}
all_splits = {}

for target in target_cols:
    print("=" * 70)
    print(f"Running models for target: {target}")
    print("=" * 70)

    results_df, trained_models, X_train, X_val, X_test, y_train, y_val, y_test = run_models_for_target(
        df, feature_cols, target
    )

    print(results_df)

    all_results[target] = results_df
    all_models[target] = trained_models
    all_splits[target] = (X_train, X_val, X_test, y_train, y_val, y_test)

In [ ]:
# Pick best model for each target based on validation RMSE
best_models = {}

for target in target_cols:
    best_model_name = all_results[target].iloc[0]["model_name"]
    best_models[target] = all_models[target][best_model_name]
    print(f"Best model for {target}: {best_model_name}")

### 4.4 Test Set Evaluation

In [ ]:
# Final test-set evaluation
final_test_results = []

for target in target_cols:
    X_train, X_val, X_test, y_train, y_val, y_test = all_splits[target]
    best_model_name = all_results[target].iloc[0]["model_name"]
    best_model = best_models[target]

    test_pred = best_model.predict(X_test)

    test_mae = mean_absolute_error(y_test, test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    test_r2 = r2_score(y_test, test_pred)

    final_test_results.append({
        "target": target,
        "best_model": best_model_name,
        "test_mae": test_mae,
        "test_rmse": test_rmse,
        "test_r2": test_r2
    })

final_test_results_df = pd.DataFrame(final_test_results)
print("\nFinal Test Results:")
print(final_test_results_df)

## 5. Module 3 — Argument Quality (Rule-Based)

Evaluates whether a paper's conclusions are justified by its results. Uses dataset signals (argument_strength, logical_consistency, overclaim) plus text-based keyword checks to generate a quality score (0–100), quality level, and diagnostic report.

In [ ]:
def argument_quality_analyzer(row):
    """
    Analyze argument quality for one paper.

    Returns dict with: argument_quality_score, argument_quality_level,
    logical_consistency_level, overclaim_risk, issues, strengths, summary.
    """

    abstract = str(row.get("abstract", "")).lower()
    results = str(row.get("results", "")).lower()

    argument_strength = float(row.get("argument_strength", 0))
    logical_consistency = float(row.get("logical_consistency", 0))
    overclaim = int(row.get("overclaim", 0))

    issues = []
    strengths = []

    # Start with weighted score from dataset signals
    score = 0
    score += argument_strength * 45
    score += logical_consistency * 45

    # Overclaim penalty
    if overclaim == 1:
        score -= 20
        issues.append("The paper may make claims that go beyond what the results support.")
    else:
        strengths.append("The claims appear reasonably restrained.")

    # Keyword groups
    strong_claim_keywords = [
        "proves", "demonstrates", "confirms", "shows that", "establishes",
        "clearly", "definitively", "significantly improves", "guarantees"
    ]

    cautious_keywords = [
        "suggest", "may", "might", "could", "appears", "indicates",
        "potentially", "associated with", "is likely"
    ]

    limitation_keywords = [
        "limitation", "limitations", "limited", "however", "although",
        "bias", "constraint", "caution"
    ]

    results_support_keywords = [
        "result", "results", "found", "observed", "measured", "analysis showed",
        "we found", "our findings", "data suggest"
    ]

    # Text-based checks
    abstract_has_strong_claims = contains_any(abstract, strong_claim_keywords)
    abstract_has_caution = contains_any(abstract, cautious_keywords)
    results_has_support = contains_any(results, results_support_keywords)
    results_has_limitations = contains_any(results, limitation_keywords)

    if abstract_has_strong_claims and overclaim == 1:
        issues.append("The abstract uses strong claim language that may not be fully justified by the results.")
        score -= 10

    if not abstract_has_caution and overclaim == 1:
        issues.append("The paper may need more cautious wording in its claims.")
        score -= 8

    if results_has_support:
        strengths.append("The results section includes evidence-oriented language.")
        score += 5
    else:
        issues.append("The results section may not clearly present supporting evidence.")
        score -= 8

    if results_has_limitations:
        strengths.append("The paper acknowledges limitations or caution in interpreting the findings.")
        score += 5
    else:
        issues.append("The paper may not clearly discuss limitations when presenting conclusions.")
        score -= 5

    # Logical consistency label
    if logical_consistency >= 0.75:
        logical_label = "Strong"
    elif logical_consistency >= 0.6:
        logical_label = "Moderate"
    else:
        logical_label = "Weak"
        issues.append("The reasoning from results to conclusion may be weak.")

    # Overclaim risk label
    if overclaim == 1:
        overclaim_risk = "High"
    elif abstract_has_strong_claims and not abstract_has_caution:
        overclaim_risk = "Moderate"
    else:
        overclaim_risk = "Low"

    # Argument quality level
    score = max(0, min(score, 100))

    if score >= 75:
        quality_level = "Strong"
    elif score >= 50:
        quality_level = "Moderate"
    else:
        quality_level = "Weak"

    # More strengths based on numeric signals
    if argument_strength >= 0.7:
        strengths.append("The paper presents a relatively strong argument.")
    elif argument_strength < 0.6:
        issues.append("The argument may not be sufficiently supported by the presented evidence.")

    if logical_consistency >= 0.7:
        strengths.append("The conclusions are mostly aligned with the reported results.")

    # Summary
    if quality_level == "Strong":
        summary = (
            "The paper's argument is generally well supported by the results, "
            "with a fairly consistent connection between evidence and conclusions."
        )
    elif quality_level == "Moderate":
        summary = (
            "The paper shows some reasonable evidence-conclusion alignment, "
            "but there are still concerns about support, caution, or consistency."
        )
    else:
        summary = (
            "The paper's argument may be weak, overstated, or insufficiently supported "
            "by the reported results."
        )

    return {
        "argument_quality_score": round(score, 2),
        "argument_quality_level": quality_level,
        "logical_consistency_level": logical_label,
        "overclaim_risk": overclaim_risk,
        "issues": issues,
        "strengths": strengths,
        "summary": summary
    }


def print_argument_quality_report(row):
    """Pretty-print a full argument quality report for one paper."""
    result = argument_quality_analyzer(row)

    print("=" * 80)
    print("ARGUMENT QUALITY REPORT")
    print("=" * 80)
    print(f"Title: {row.get('title', '')}")
    print(f"\nArgument Quality Score: {result['argument_quality_score']}/100")
    print(f"Argument Quality Level: {result['argument_quality_level']}")
    print(f"Logical Consistency: {result['logical_consistency_level']}")
    print(f"Overclaim Risk: {result['overclaim_risk']}")

    print("\nStrengths:")
    if result["strengths"]:
        for s in result["strengths"]:
            print(f"- {s}")
    else:
        print("- No major strengths detected.")

    print("\nIssues:")
    if result["issues"]:
        for issue in result["issues"]:
            print(f"- {issue}")
    else:
        print("- No major argument issues detected.")

    print("\nSummary:")
    print(result["summary"])
    print("=" * 80)

In [ ]:
# Apply to every row
argument_outputs = df.apply(argument_quality_analyzer, axis=1)

df["argument_quality_score_final"] = argument_outputs.apply(lambda x: x["argument_quality_score"])
df["argument_quality_level"] = argument_outputs.apply(lambda x: x["argument_quality_level"])
df["logical_consistency_level"] = argument_outputs.apply(lambda x: x["logical_consistency_level"])
df["overclaim_risk"] = argument_outputs.apply(lambda x: x["overclaim_risk"])
df["argument_issues"] = argument_outputs.apply(lambda x: " | ".join(x["issues"]))
df["argument_strengths_text"] = argument_outputs.apply(lambda x: " | ".join(x["strengths"]))
df["argument_summary"] = argument_outputs.apply(lambda x: x["summary"])


# Preview results
print("=" * 80)
print("Argument Quality Module Preview")
print("=" * 80)
print(df[[
    "title",
    "argument_quality_score_final",
    "argument_quality_level",
    "logical_consistency_level",
    "overclaim_risk",
    "argument_summary"
]].head())

# Example
print_argument_quality_report(df.iloc[0])

## 6. Module 4 — Peer Review Quality Predictor (Rule-Based)

Acts as a lightweight mock peer reviewer. Uses paper-level signals (citation need, overclaim, argument strength, logical consistency, readability, jargon, structure, and issue flags) to generate a reviewer concern score (0–100), concern level, strengths, weaknesses, missing elements, and a summary.

This module is rule-based because some reviewer-related labels (missing_element, reviewer_concern) appear as one class only in the dataset, making supervised learning infeasible.

In [ ]:
def peer_review_predictor(row):
    """
    Generate a lightweight peer review report for one paper.

    Returns dict with: reviewer_concern_score, reviewer_concern_level,
    strengths, weaknesses, missing_elements, summary.
    """

    strengths = []
    weaknesses = []
    missing_elements = []
    concern_score = 0

    # Collect paper text
    title = str(row.get("title", ""))
    abstract = str(row.get("abstract", ""))
    results = str(row.get("results", ""))

    # Combine all text for keyword checks
    full_text = f"{title} {abstract} {results}".lower()

    # Collect numeric / flag inputs
    citation_needed = int(row.get("citation_needed", 0))
    overclaim = int(row.get("overclaim", 0))
    argument_strength = float(row.get("argument_strength", 0))
    logical_consistency = float(row.get("logical_consistency", 0))
    readability_score = float(row.get("readability_score", 0))
    jargon_density = float(row.get("jargon_density", 0))
    structural_complexity = float(row.get("structural_complexity", 0))
    title_issues_flag = int(row.get("title_issues_flag", 0))
    abstract_issues_flag = int(row.get("abstract_issues_flag", 0))


    # Strength detection
    if argument_strength >= 0.7:
        strengths.append("The paper presents a relatively strong argument supported by its results.")

    if logical_consistency >= 0.7:
        strengths.append("The conclusions appear logically connected to the reported results.")

    if readability_score >= 0.6:
        strengths.append("The writing is reasonably readable and easy to follow.")

    if jargon_density <= 0.5:
        strengths.append("The paper avoids excessive jargon, which improves clarity.")

    if structural_complexity <= 0.5:
        strengths.append("The structure is not overly complex, which may help readability.")

    if overclaim == 0:
        strengths.append("The claims appear fairly restrained and not obviously overstated.")

    if title_issues_flag == 0 and abstract_issues_flag == 0:
        strengths.append("The title and abstract appear reasonably aligned with the paper content.")


    # Weakness detection + scoring
    if citation_needed == 1:
        weaknesses.append("Some claims may require stronger citation support.")
        concern_score += 20

    if overclaim == 1:
        weaknesses.append("The paper may overstate its findings or generalize beyond the results.")
        concern_score += 20

    if argument_strength < 0.6:
        weaknesses.append("The argument may not be fully convincing or sufficiently supported.")
        concern_score += 15

    if logical_consistency < 0.6:
        weaknesses.append("The link between results and conclusion may be weak or incomplete.")
        concern_score += 15

    if readability_score < 0.5:
        weaknesses.append("The writing may be difficult to read and follow.")
        concern_score += 10

    if jargon_density > 0.7:
        weaknesses.append("The paper uses dense technical language that may increase reviewer burden.")
        concern_score += 10

    if structural_complexity > 0.7:
        weaknesses.append("The paper structure appears complex, which may reduce clarity.")
        concern_score += 10

    if title_issues_flag == 1:
        weaknesses.append("The title may contain wording that reviewers could question.")
        concern_score += 8

    if abstract_issues_flag == 1:
        weaknesses.append("The abstract may omit important context, caution, or limitations.")
        concern_score += 8


    # Missing element detection
    limitation_keywords = [
        "limitation", "limitations", "limited", "constraint", "constraints",
        "weakness", "weaknesses", "caution", "bias", "biases"
    ]

    future_work_keywords = [
        "future work", "future research", "further work", "further research",
        "next step", "next steps", "in future"
    ]

    evidence_keywords = [
        "previous work", "prior work", "previous studies", "prior studies",
        "according to", "as reported", "literature", "studies have shown"
    ]

    hedging_keywords = [
        "may", "might", "could", "suggest", "possibly", "appears", "likely",
        "indicate", "can"
    ]

    if not contains_any(full_text, limitation_keywords):
        missing_elements.append("Explicit discussion of limitations may be missing.")
        concern_score += 10

    if not contains_any(full_text, future_work_keywords):
        missing_elements.append("Future work or next-step discussion may be missing.")
        concern_score += 5

    if citation_needed == 1 and not contains_any(full_text, evidence_keywords):
        missing_elements.append("Evidence framing or reference to related work may be insufficient.")
        concern_score += 8

    if overclaim == 1 and not contains_any(full_text, hedging_keywords):
        missing_elements.append("The paper may need more cautious scientific wording.")
        concern_score += 7


    # Normalize score to 0-100
    concern_score = min(concern_score, 100)

    # Concern level
    if concern_score >= 70:
        concern_level = "High"
    elif concern_score >= 40:
        concern_level = "Moderate"
    else:
        concern_level = "Low"

    # Summary generation
    if concern_level == "High":
        summary = (
            "This paper shows several issues that reviewers may question, "
            "especially around evidence support, claim strength, clarity, "
            "or missing discussion."
        )
    elif concern_level == "Moderate":
        summary = (
            "This paper has some promising aspects, but reviewers may still "
            "raise concerns about support, wording, clarity, or completeness."
        )
    else:
        summary = (
            "This paper appears relatively balanced overall, with fewer likely "
            "reviewer concerns."
        )

    return {
        "reviewer_concern_score": concern_score,
        "reviewer_concern_level": concern_level,
        "strengths": strengths,
        "weaknesses": weaknesses,
        "missing_elements": missing_elements,
        "summary": summary
    }


def print_peer_review_report(row):
    """Pretty-print a full peer review report for one paper."""
    result = peer_review_predictor(row)

    print("=" * 80)
    print("PEER REVIEW QUALITY PREDICTOR REPORT")
    print("=" * 80)
    print(f"Title: {row.get('title', '')}")
    print(f"\nReviewer Concern Score: {result['reviewer_concern_score']}/100")
    print(f"Concern Level: {result['reviewer_concern_level']}")

    print("\nStrengths:")
    if result["strengths"]:
        for s in result["strengths"]:
            print(f"- {s}")
    else:
        print("- No strong strengths detected.")

    print("\nWeaknesses:")
    if result["weaknesses"]:
        for w in result["weaknesses"]:
            print(f"- {w}")
    else:
        print("- No major weaknesses detected.")

    print("\nMissing Elements:")
    if result["missing_elements"]:
        for m in result["missing_elements"]:
            print(f"- {m}")
    else:
        print("- No obvious missing elements detected.")

    print("\nSummary:")
    print(result["summary"])
    print("=" * 80)

In [ ]:
# Apply peer review predictor to every row
peer_outputs = df.apply(peer_review_predictor, axis=1)

df["peer_review_score"] = peer_outputs.apply(lambda x: x["reviewer_concern_score"])
df["peer_review_level"] = peer_outputs.apply(lambda x: x["reviewer_concern_level"])
df["peer_review_strengths"] = peer_outputs.apply(lambda x: " | ".join(x["strengths"]))
df["peer_review_weaknesses"] = peer_outputs.apply(lambda x: " | ".join(x["weaknesses"]))
df["peer_review_missing_elements"] = peer_outputs.apply(lambda x: " | ".join(x["missing_elements"]))
df["peer_review_summary"] = peer_outputs.apply(lambda x: x["summary"])


# Preview
print("=" * 80)
print("Peer Review Predictor Preview")
print("=" * 80)
print(df[[
    "title",
    "peer_review_score",
    "peer_review_level",
    "peer_review_summary"
]].head())

# Example
print_peer_review_report(df.iloc[0])

# Distribution
print("\nReviewer Concern Level Distribution:")
print(df["peer_review_level"].value_counts())

## 7. Module 5 — Headline Accuracy Checker

Evaluates whether a paper's title accurately reflects its findings or exaggerates them. Includes both a binary classifier (accurate vs exaggerated) and a regression model for exaggeration severity score.

### 7.1 Feature Engineering

Note: This module works on the shared `df` (already cleaned and preprocessed). We add headline-specific features on top of the existing columns.

In [ ]:
# Hand-crafted lexical cue vocabularies
HEADLINE_ABSOLUTE_WORDS = [
    "all", "always", "never", "prove", "proves", "definitive", "definitively",
    "clear", "clearly", "guarantee", "guarantees", "establish", "establishes",
    "confirm", "confirms"
]

HEADLINE_CAUSAL_WORDS = [
    "cause", "causes", "caused", "causal",
    "drive", "drives", "driven by",
    "lead to", "leads to",
    "result in", "results in"
]

HEADLINE_HEDGE_WORDS = [
    "may", "might", "could", "suggest", "suggests",
    "appears", "likely", "potentially", "associated with"
]

HEADLINE_LIMITATION_WORDS = [
    "limitation", "limitations", "limited", "however",
    "although", "caution", "bias", "constraint", "future work"
]

# Combine title + abstract + results into one text field
df["headline_input_text"] = df["title"] + " [SEP] " + df["abstract"] + " [SEP] " + df["results"]

# Interpretable numeric features
df["title_abs_word_count"] = df["title"].apply(lambda s: count_keywords(s, HEADLINE_ABSOLUTE_WORDS))
df["abstract_abs_word_count"] = df["abstract"].apply(lambda s: count_keywords(s, HEADLINE_ABSOLUTE_WORDS))
df["title_causal_word_count"] = df["title"].apply(lambda s: count_keywords(s, HEADLINE_CAUSAL_WORDS))
df["abstract_causal_word_count"] = df["abstract"].apply(lambda s: count_keywords(s, HEADLINE_CAUSAL_WORDS))
df["results_limitation_count"] = df["results"].apply(lambda s: count_keywords(s, HEADLINE_LIMITATION_WORDS))
df["abstract_hedge_count"] = df["abstract"].apply(lambda s: count_keywords(s, HEADLINE_HEDGE_WORDS))
df["title_hedge_count"] = df["title"].apply(lambda s: count_keywords(s, HEADLINE_HEDGE_WORDS))

# Binary target: 0 = Accurate, 1 = Exaggerated / potentially misleading
df["headline_exaggerated"] = (
    df["headline_verdict"].astype(str).str.strip().str.lower() != "accurate"
).astype(int)

headline_feature_cols = [
    "headline_input_text",
    "title_length",
    "abstract_length",
    "results_length",
    "title_issues_flag",
    "abstract_issues_flag",
    "title_abs_word_count",
    "abstract_abs_word_count",
    "title_causal_word_count",
    "abstract_causal_word_count",
    "results_limitation_count",
    "abstract_hedge_count",
    "title_hedge_count"
]

X_headline = df[headline_feature_cols]
y_headline_bin = df["headline_exaggerated"]
y_headline_score = df["exaggeration_score"]  # already normalized to 0-1 from Section 1

print("Headline verdict distribution:")
print(df["headline_verdict"].value_counts())
print("\nBinary target distribution:")
print(y_headline_bin.value_counts())

### 7.2 Train / Val / Test Split

In [ ]:
X_hl_train_val, X_hl_test, y_hl_train_val, y_hl_test = train_test_split(
    X_headline, y_headline_bin, test_size=0.20, random_state=RANDOM_STATE, stratify=y_headline_bin
)

X_hl_train, X_hl_val, y_hl_train, y_hl_val = train_test_split(
    X_hl_train_val, y_hl_train_val, test_size=0.50, random_state=RANDOM_STATE, stratify=y_hl_train_val
)

# Keep regression target aligned by index
y_hl_score_train_val = y_headline_score.loc[X_hl_train_val.index]
y_hl_score_test = y_headline_score.loc[X_hl_test.index]
y_hl_score_train = y_headline_score.loc[X_hl_train.index]
y_hl_score_val = y_headline_score.loc[X_hl_val.index]

print("Split sizes:")
print(f"Train: {len(X_hl_train)} | Val: {len(X_hl_val)} | Test: {len(X_hl_test)}")

### 7.3 Pipeline Builders

In [ ]:
def build_headline_pipeline(model):
    """Build a TF-IDF + numeric feature pipeline for headline tasks."""
    text_feature = "headline_input_text"
    numeric_features = [c for c in headline_feature_cols if c != text_feature]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "text",
                TfidfVectorizer(
                    max_features=6000,
                    ngram_range=(1, 2),
                    min_df=2,
                    stop_words="english",
                    sublinear_tf=True
                ),
                text_feature
            ),
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_features
            )
        ]
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

### 7.4 Binary Headline Accuracy Classifier

In [ ]:
print("=" * 70)
print("Binary Headline Accuracy Classifier")
print("=" * 70)

clf = build_headline_pipeline(
    LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1.0
    )
)

clf.fit(X_hl_train, y_hl_train)

# Threshold tuning on validation set
val_proba = clf.predict_proba(X_hl_val)[:, 1]
best_threshold = 0.50
best_val_f1 = -1

for thr in np.arange(0.30, 0.71, 0.05):
    val_pred_thr = (val_proba >= thr).astype(int)
    val_f1 = f1_score(y_hl_val, val_pred_thr, average="macro")
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_threshold = float(thr)

print(f"Best validation threshold: {best_threshold:.2f}")
print(f"Best validation macro F1: {best_val_f1:.4f}")

# Refit on train + validation before final test
clf_final = build_headline_pipeline(
    LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1.0
    )
)
clf_final.fit(X_hl_train_val, y_hl_train_val)

test_proba = clf_final.predict_proba(X_hl_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

print("\nBinary Test Results:")
print(f"Accuracy      : {accuracy_score(y_hl_test, test_pred):.4f}")
print(f"Precision     : {precision_score(y_hl_test, test_pred):.4f}")
print(f"Recall        : {recall_score(y_hl_test, test_pred):.4f}")
print(f"F1 (macro)    : {f1_score(y_hl_test, test_pred, average='macro'):.4f}")
print(f"F1 (weighted) : {f1_score(y_hl_test, test_pred, average='weighted'):.4f}")
print(f"ROC-AUC       : {roc_auc_score(y_hl_test, test_proba):.4f}")
print(f"PR-AUC        : {average_precision_score(y_hl_test, test_proba):.4f}")

print("\nClassification report:")
print(classification_report(
    y_hl_test,
    test_pred,
    target_names=["Accurate", "Exaggerated"]
))

print("Confusion matrix:")
print(confusion_matrix(y_hl_test, test_pred))

### 7.5 Exaggeration Severity Regressor

In [ ]:
print("=" * 70)
print("Exaggeration Severity Regressor")
print("=" * 70)

reg_models = {
    "Ridge": build_headline_pipeline(Ridge(alpha=1.0)),
    "RandomForest": build_headline_pipeline(
        RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )
    )
}

reg_results = []
trained_reg_models = {}

for name, model in reg_models.items():
    model.fit(X_hl_train, y_hl_score_train)
    val_pred = np.clip(model.predict(X_hl_val), 0, 1)

    rmse = np.sqrt(mean_squared_error(y_hl_score_val, val_pred))
    mae = mean_absolute_error(y_hl_score_val, val_pred)
    r2 = r2_score(y_hl_score_val, val_pred)

    reg_results.append({
        "model_name": name,
        "val_mae": mae,
        "val_rmse": rmse,
        "val_r2": r2
    })
    trained_reg_models[name] = model

reg_results_df = pd.DataFrame(reg_results).sort_values(by="val_rmse")
print(reg_results_df.to_string(index=False))

best_reg_name = reg_results_df.iloc[0]["model_name"]
print(f"\nBest regression model: {best_reg_name}")

best_reg_final = reg_models[best_reg_name]
best_reg_final.fit(X_hl_train_val, y_hl_score_train_val)

test_score_pred = np.clip(best_reg_final.predict(X_hl_test), 0, 1)

print("\nRegression Test Results:")
print(f"MAE  : {mean_absolute_error(y_hl_score_test, test_score_pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_hl_score_test, test_score_pred)):.4f}")
print(f"R²   : {r2_score(y_hl_score_test, test_score_pred):.4f}")

print("\nDone.")

## 8. Unified Paper Quality Model

Combines outputs from all five modules into a single weighted quality score (0–100) per paper. Each module's output is normalized to a common 0–100 scale where higher = better quality.

In [ ]:
# =====================================================================
# UNIFIED PAPER QUALITY MODEL
# =====================================================================
# This cell combines outputs from all five modules into a single
# unified assessment per paper:
#
#   Module 1: Citation Worthiness      (binary classification)
#   Module 2: Cognitive Load Estimator (regression × 3 targets)
#   Module 3: Argument Quality         (rule-based, 0-100)
#   Module 4: Peer Review Predictor    (rule-based, 0-100)
#   Module 5: Headline Accuracy        (classification + regression)
#
# Final outputs per paper:
#   - unified_quality_score (0-100)
#   - unified_quality_level (Poor / Fair / Good / Excellent)
#   - per-module sub-scores (normalized to 0-100)
#   - combined strengths, weaknesses, and recommendations
#   - human-readable unified report
# =====================================================================

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------
# 1) COLLECT MODULE OUTPUTS INTO NORMALIZED SUB-SCORES (0-100)
# ---------------------------------------------------------------------
# Each module produces outputs on different scales. We normalize
# every module to a 0-100 "quality" score where higher = better paper.

def compute_citation_score(row):
    """
    Module 1: Citation Worthiness
    If citation_needed == 1 AND the paper lacks citation support cues,
    that's a quality concern. We invert: papers that don't need extra
    citations score higher.
    """
    citation_needed = int(row.get("citation_needed", 0))
    citation_raw = float(row.get("citation_score", 50))

    # citation_score in the dataset is 0-100; higher = more likely needs citation
    # Invert so higher = better (paper already well-cited)
    score = (1 - citation_raw) * 100

    # Additional penalty if the model flags citation needed
    if citation_needed == 1:
        score = max(0, score - 15)

    return round(np.clip(score, 0, 100), 2)


def compute_cognitive_load_score(row):
    """
    Module 2: Cognitive Load Estimator
    Combines readability, jargon density, and structural complexity.
    Higher readability = good; lower jargon/complexity = good.
    All raw values are on 0-1 scale in the dataset.
    """
    readability = float(row.get("readability_score", 0.5))
    jargon = float(row.get("jargon_density", 0.5))
    complexity = float(row.get("structural_complexity", 0.5))

    # Readability: higher is better -> scale directly
    read_component = readability * 100

    # Jargon: lower is better -> invert
    jargon_component = (1 - jargon) * 100

    # Complexity: lower is better -> invert
    complexity_component = (1 - complexity) * 100

    # Weighted combination (readability matters most)
    score = (0.45 * read_component +
             0.30 * jargon_component +
             0.25 * complexity_component)

    return round(np.clip(score, 0, 100), 2)


def compute_argument_quality_score(row):
    """
    Module 3: Argument Quality (already 0-100 from the rule-based module)
    """
    return float(row.get("argument_quality_score_final", 50))


def compute_peer_review_score(row):
    """
    Module 4: Peer Review Predictor
    reviewer_concern_score is 0-100 where higher = more concerns.
    Invert so higher = better quality.
    """
    concern = float(row.get("peer_review_score", 50))
    return round(np.clip(100 - concern, 0, 100), 2)


def compute_headline_accuracy_score(row):
    """
    Module 5: Headline Accuracy Checker
    Uses exaggeration_score (0-100, higher = more exaggerated).
    Invert so higher = more accurate headline.
    """
    exaggeration = float(row.get("exaggeration_score", 50))
    headline_verdict = str(row.get("headline_verdict", "")).strip().lower()

    score = score = (1 - exaggeration) * 100

    # Bonus for accurate verdict, penalty for exaggerated
    if headline_verdict == "accurate":
        score = min(100, score + 10)
    elif headline_verdict in ["exaggerated", "misleading"]:
        score = max(0, score - 10)

    return round(np.clip(score, 0, 100), 2)


# ---------------------------------------------------------------------
# 2) UNIFIED QUALITY SCORE (WEIGHTED COMBINATION)
# ---------------------------------------------------------------------
# Weights reflect relative importance for overall paper quality.
# These can be adjusted based on domain priorities.

MODULE_WEIGHTS = {
    "citation":          0.15,   # Citation support
    "cognitive_load":    0.15,   # Readability / clarity
    "argument_quality":  0.30,   # Core: are claims justified?
    "peer_review":       0.25,   # Simulated reviewer concerns
    "headline_accuracy": 0.15    # Title vs. content alignment
}

def compute_unified_score(row):
    """
    Compute the weighted unified paper quality score.
    Returns a dict with all sub-scores and the final unified score.
    """
    sub_scores = {
        "citation_score":          compute_citation_score(row),
        "cognitive_load_score":    compute_cognitive_load_score(row),
        "argument_quality_score":  compute_argument_quality_score(row),
        "peer_review_score":       compute_peer_review_score(row),
        "headline_accuracy_score": compute_headline_accuracy_score(row)
    }

    unified = (
        MODULE_WEIGHTS["citation"]          * sub_scores["citation_score"] +
        MODULE_WEIGHTS["cognitive_load"]    * sub_scores["cognitive_load_score"] +
        MODULE_WEIGHTS["argument_quality"]  * sub_scores["argument_quality_score"] +
        MODULE_WEIGHTS["peer_review"]       * sub_scores["peer_review_score"] +
        MODULE_WEIGHTS["headline_accuracy"] * sub_scores["headline_accuracy_score"]
    )

    sub_scores["unified_quality_score"] = round(np.clip(unified, 0, 100), 2)

    # Quality level
    if unified >= 75:
        sub_scores["unified_quality_level"] = "Excellent"
    elif unified >= 60:
        sub_scores["unified_quality_level"] = "Good"
    elif unified >= 40:
        sub_scores["unified_quality_level"] = "Fair"
    else:
        sub_scores["unified_quality_level"] = "Poor"

    return sub_scores


# ---------------------------------------------------------------------
# 3) APPLY TO FULL DATASET
# ---------------------------------------------------------------------
print("=" * 80)
print("  UNIFIED PAPER QUALITY MODEL")
print("=" * 80)

unified_outputs = df.apply(compute_unified_score, axis=1)

# Add unified columns to df
df["unified_citation_score"]     = unified_outputs.apply(lambda x: x["citation_score"])
df["unified_cognitive_score"]    = unified_outputs.apply(lambda x: x["cognitive_load_score"])
df["unified_argument_score"]     = unified_outputs.apply(lambda x: x["argument_quality_score"])
df["unified_peer_review_score"]  = unified_outputs.apply(lambda x: x["peer_review_score"])
df["unified_headline_score"]     = unified_outputs.apply(lambda x: x["headline_accuracy_score"])
df["unified_quality_score"]      = unified_outputs.apply(lambda x: x["unified_quality_score"])
df["unified_quality_level"]      = unified_outputs.apply(lambda x: x["unified_quality_level"])


# ---------------------------------------------------------------------
# 4) SUMMARY STATISTICS
# ---------------------------------------------------------------------
print("\nModule Weight Distribution:")
for module, weight in MODULE_WEIGHTS.items():
    print(f"  {module:25s} -> {weight:.0%}")

print(f"\n{'─' * 60}")
print("Sub-Score Statistics (0-100, higher = better):")
print(f"{'─' * 60}")
score_cols = [
    ("Citation Support",    "unified_citation_score"),
    ("Cognitive Load",      "unified_cognitive_score"),
    ("Argument Quality",    "unified_argument_score"),
    ("Peer Review",         "unified_peer_review_score"),
    ("Headline Accuracy",   "unified_headline_score"),
    ("UNIFIED SCORE",       "unified_quality_score")
]

for label, col in score_cols:
    vals = df[col]
    print(f"  {label:22s}  mean={vals.mean():6.2f}  std={vals.std():6.2f}  "
          f"min={vals.min():6.2f}  max={vals.max():6.2f}")

print(f"\n{'─' * 60}")
print("Quality Level Distribution:")
print(f"{'─' * 60}")
level_counts = df["unified_quality_level"].value_counts()
for level in ["Excellent", "Good", "Fair", "Poor"]:
    count = level_counts.get(level, 0)
    pct = count / len(df) * 100
    bar = "" * int(pct / 2)
    print(f"  {level:10s}  {count:4d} ({pct:5.1f}%)  {bar}")


# ---------------------------------------------------------------------
# 5) UNIFIED REPORT GENERATOR
# ---------------------------------------------------------------------
def generate_unified_report(row):
    """
    Generates a comprehensive human-readable quality report for one paper
    by combining outputs from all five modules.
    """
    scores = compute_unified_score(row)

    # Collect issues and strengths from rule-based modules
    arg_result = argument_quality_analyzer(row)
    review_result = peer_review_predictor(row)

    report = []
    report.append("=" * 80)
    report.append("UNIFIED PAPER QUALITY REPORT")
    report.append("=" * 80)
    report.append(f"Title: {row.get('title', 'N/A')}")
    report.append(f"Year:  {row.get('year', 'N/A')}")

    # Overall score
    report.append(f"\n{'─' * 60}")
    report.append(f"  OVERALL QUALITY SCORE: {scores['unified_quality_score']}/100  "
                  f"[{scores['unified_quality_level']}]")
    report.append(f"{'─' * 60}")

    # Sub-scores breakdown
    report.append("\nModule Breakdown:")
    module_labels = [
        ("Citation Support",    "citation_score",          MODULE_WEIGHTS["citation"]),
        ("Cognitive Load",      "cognitive_load_score",    MODULE_WEIGHTS["cognitive_load"]),
        ("Argument Quality",    "argument_quality_score",  MODULE_WEIGHTS["argument_quality"]),
        ("Peer Review",         "peer_review_score",       MODULE_WEIGHTS["peer_review"]),
        ("Headline Accuracy",   "headline_accuracy_score", MODULE_WEIGHTS["headline_accuracy"])
    ]

    for label, key, weight in module_labels:
        val = scores[key]
        bar = " " * int(val / 5) + " " * (20 - int(val / 5))
        report.append(f"  {label:22s}  {bar}  {val:5.1f}/100  (weight: {weight:.0%})")

    # Combined strengths
    all_strengths = arg_result.get("strengths", []) + review_result.get("strengths", [])
    if all_strengths:
        report.append(f"\n{'─' * 60}")
        report.append("Strengths:")
        for s in list(dict.fromkeys(all_strengths)):  # deduplicate preserving order
            report.append(f"  ✓ {s}")

    # Combined weaknesses / issues
    all_issues = arg_result.get("issues", []) + review_result.get("weaknesses", [])
    if all_issues:
        report.append(f"\n{'─' * 60}")
        report.append("Issues & Concerns:")
        for issue in list(dict.fromkeys(all_issues)):
            report.append(f"  ✗ {issue}")

    # Missing elements
    missing = review_result.get("missing_elements", [])
    if missing:
        report.append(f"\n{'─' * 60}")
        report.append("Potentially Missing Elements:")
        for m in missing:
            report.append(f"  ? {m}")

    # Recommendations
    report.append(f"\n{'─' * 60}")
    report.append("Recommendations:")
    if scores["citation_score"] < 50:
        report.append("  → Strengthen citation support for key claims.")
    if scores["cognitive_load_score"] < 50:
        report.append("  → Improve readability; consider reducing jargon or simplifying structure.")
    if scores["argument_quality_score"] < 50:
        report.append("  → Revisit argument flow; ensure conclusions follow from evidence.")
    if scores["peer_review_score"] < 50:
        report.append("  → Address likely reviewer concerns before submission.")
    if scores["headline_accuracy_score"] < 50:
        report.append("  → Revise the title to better reflect actual findings.")
    if scores["unified_quality_score"] >= 75:
        report.append("  → Paper is in strong shape overall. Minor polish may still help.")

    report.append("=" * 80)

    return "\n".join(report)


# ---------------------------------------------------------------------
# 6) PRINT SAMPLE REPORTS
# ---------------------------------------------------------------------
# Show reports for papers at different quality levels
print("\n\n")

# Best paper
best_idx = df["unified_quality_score"].idxmax()
print(generate_unified_report(df.loc[best_idx]))

print("\n")

# Median paper
median_score = df["unified_quality_score"].median()
median_idx = (df["unified_quality_score"] - median_score).abs().idxmin()
print(generate_unified_report(df.loc[median_idx]))

print("\n")

# Weakest paper
worst_idx = df["unified_quality_score"].idxmin()
print(generate_unified_report(df.loc[worst_idx]))


# ---------------------------------------------------------------------
# 7) SAVE UNIFIED RESULTS
# ---------------------------------------------------------------------
output_cols = [
    "id", "title", "year",
    "unified_citation_score",
    "unified_cognitive_score",
    "unified_argument_score",
    "unified_peer_review_score",
    "unified_headline_score",
    "unified_quality_score",
    "unified_quality_level",
    # Include key sub-module outputs for reference
    "argument_quality_level",
    "logical_consistency_level",
    "overclaim_risk",
    "peer_review_level",
    "argument_summary",
    "peer_review_summary"
]

# Only include columns that exist in df
output_cols = [c for c in output_cols if c in df.columns]
unified_df = df[output_cols].copy()

unified_df.to_csv("unified_paper_quality_results.csv", index=False)
print(f"\nSaved unified results → unified_paper_quality_results.csv")
print(f"Total papers assessed: {len(unified_df)}")
print(f"Columns saved: {len(output_cols)}")

## 9. Evaluation

This section consolidates all module results into a single evaluation framework with two parts:

1. **Quantitative evaluation** — accuracy, precision, recall, F1-score for classification modules; MAE, RMSE, R² for regression modules; and distribution analysis for rule-based modules.
2. **Qualitative evaluation** — sample reports at different quality levels, cross-module agreement analysis, and case studies.

### 9.1 Quantitative Evaluation

#### 9.1.1 Classification Metrics — Citation Worthiness (Module 1)

The citation module was evaluated using 5-fold stratified cross-validation, comparing Logistic Regression (TF-IDF + handcrafted features) against SciBERT + numeric features.

In [ ]:
print("=" * 70)
print("MODULE 1: Citation Worthiness — Classification Metrics")
print("=" * 70)

# Per-fold results
print("\nLogistic Regression (per fold):")
print(lr_df.to_string(index=False))
print(f"\n  Mean Accuracy : {lr_df['accuracy'].mean():.4f} ± {lr_df['accuracy'].std():.4f}")
print(f"  Mean F1       : {lr_df['f1'].mean():.4f} ± {lr_df['f1'].std():.4f}")

print("\nSciBERT + Features (per fold):")
print(scibert_df.to_string(index=False))
print(f"\n  Mean Accuracy : {scibert_df['accuracy'].mean():.4f} ± {scibert_df['accuracy'].std():.4f}")
print(f"  Mean F1       : {scibert_df['f1'].mean():.4f} ± {scibert_df['f1'].std():.4f}")

# Summary comparison table
citation_summary = pd.DataFrame({
    "Model": ["Logistic Regression", "SciBERT + Features"],
    "Mean Accuracy": [lr_df["accuracy"].mean(), scibert_df["accuracy"].mean()],
    "Std Accuracy": [lr_df["accuracy"].std(), scibert_df["accuracy"].std()],
    "Mean F1": [lr_df["f1"].mean(), scibert_df["f1"].mean()],
    "Std F1": [lr_df["f1"].std(), scibert_df["f1"].std()],
})
print("\nCitation Module — Summary:")
print(citation_summary.to_string(index=False))

best_citation_model = "SciBERT + Features" if scibert_df["f1"].mean() >= lr_df["f1"].mean() else "Logistic Regression"
print(f"\nBest model: {best_citation_model}")

#### 9.1.2 Regression Metrics — Cognitive Load (Module 2)

Three regression targets (readability, jargon density, structural complexity) were each evaluated with Ridge and Random Forest regressors.

In [ ]:
print("=" * 70)
print("MODULE 2: Cognitive Load — Regression Metrics")
print("=" * 70)

print("\nTest Set Results (Best Model per Target):")
print(final_test_results_df.to_string(index=False))

print(f"\n  Average MAE  : {final_test_results_df['test_mae'].mean():.4f}")
print(f"  Average RMSE : {final_test_results_df['test_rmse'].mean():.4f}")
print(f"  Average R²   : {final_test_results_df['test_r2'].mean():.4f}")

#### 9.1.3 Classification + Regression Metrics — Headline Accuracy (Module 5)

The headline module includes both a binary classifier (accurate vs exaggerated) and a regression model for exaggeration severity.

In [ ]:
print("=" * 70)
print("MODULE 5: Headline Accuracy — Classification Metrics")
print("=" * 70)

headline_clf_metrics = {
    "Accuracy": accuracy_score(y_hl_test, test_pred),
    "Precision": precision_score(y_hl_test, test_pred),
    "Recall": recall_score(y_hl_test, test_pred),
    "F1 (macro)": f1_score(y_hl_test, test_pred, average="macro"),
    "F1 (weighted)": f1_score(y_hl_test, test_pred, average="weighted"),
    "ROC-AUC": roc_auc_score(y_hl_test, test_proba),
    "PR-AUC": average_precision_score(y_hl_test, test_proba),
}

for metric, value in headline_clf_metrics.items():
    print(f"  {metric:15s} : {value:.4f}")

print(f"\nOptimal threshold: {best_threshold:.2f}")

print("\n" + "=" * 70)
print("MODULE 5: Headline Accuracy — Regression Metrics")
print("=" * 70)

headline_reg_metrics = {
    "MAE": mean_absolute_error(y_hl_score_test, test_score_pred),
    "RMSE": np.sqrt(mean_squared_error(y_hl_score_test, test_score_pred)),
    "R²": r2_score(y_hl_score_test, test_score_pred),
}

for metric, value in headline_reg_metrics.items():
    print(f"  {metric:15s} : {value:.4f}")

#### 9.1.4 Rule-Based Module Distributions (Modules 3 & 4)

Since the argument quality and peer review modules are rule-based, we evaluate them by analyzing the distribution and consistency of their outputs rather than traditional ML metrics.

In [ ]:
print("=" * 70)
print("MODULE 3: Argument Quality — Distribution Analysis")
print("=" * 70)

print("\nQuality Level Distribution:")
aq_dist = df["argument_quality_level"].value_counts()
for level in ["Strong", "Moderate", "Weak"]:
    count = aq_dist.get(level, 0)
    pct = count / len(df) * 100
    print(f"  {level:10s}  {count:4d} ({pct:5.1f}%)")

print(f"\nArgument Quality Score Statistics:")
print(f"  Mean  : {df['argument_quality_score_final'].mean():.2f}")
print(f"  Std   : {df['argument_quality_score_final'].std():.2f}")
print(f"  Min   : {df['argument_quality_score_final'].min():.2f}")
print(f"  Max   : {df['argument_quality_score_final'].max():.2f}")

print(f"\nOverclaim Risk Distribution:")
print(df["overclaim_risk"].value_counts().to_string())

print(f"\nLogical Consistency Level Distribution:")
print(df["logical_consistency_level"].value_counts().to_string())

print("\n" + "=" * 70)
print("MODULE 4: Peer Review Predictor — Distribution Analysis")
print("=" * 70)

print("\nConcern Level Distribution:")
pr_dist = df["peer_review_level"].value_counts()
for level in ["Low", "Moderate", "High"]:
    count = pr_dist.get(level, 0)
    pct = count / len(df) * 100
    print(f"  {level:10s}  {count:4d} ({pct:5.1f}%)")

print(f"\nPeer Review Score Statistics:")
print(f"  Mean  : {df['peer_review_score'].mean():.2f}")
print(f"  Std   : {df['peer_review_score'].std():.2f}")
print(f"  Min   : {df['peer_review_score'].min():.2f}")
print(f"  Max   : {df['peer_review_score'].max():.2f}")

#### 9.1.5 Unified Model — Overall Score Distribution

In [ ]:
print("=" * 70)
print("UNIFIED MODEL — Score Distribution")
print("=" * 70)

# Quality level breakdown
print("\nUnified Quality Level Distribution:")
uq_dist = df["unified_quality_level"].value_counts()
for level in ["Excellent", "Good", "Fair", "Poor"]:
    count = uq_dist.get(level, 0)
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"  {level:10s}  {count:4d} ({pct:5.1f}%)  {bar}")

# Per-module sub-score summary
print("\nSub-Score Summary (0-100, higher = better):")
print("-" * 65)
sub_score_cols = {
    "Citation Support":    "unified_citation_score",
    "Cognitive Load":      "unified_cognitive_score",
    "Argument Quality":    "unified_argument_score",
    "Peer Review":         "unified_peer_review_score",
    "Headline Accuracy":   "unified_headline_score",
    "UNIFIED (weighted)":  "unified_quality_score",
}
for label, col in sub_score_cols.items():
    vals = df[col]
    print(f"  {label:22s}  mean={vals.mean():6.2f}  std={vals.std():6.2f}  "
          f"min={vals.min():6.2f}  max={vals.max():6.2f}")

#### 9.1.6 Cross-Module Correlation

How do the five module scores relate to each other? Strong correlations indicate consistent quality signals; weak correlations suggest each module captures a different dimension.

In [ ]:
print("=" * 70)
print("Cross-Module Score Correlations")
print("=" * 70)

corr_cols = [
    "unified_citation_score",
    "unified_cognitive_score",
    "unified_argument_score",
    "unified_peer_review_score",
    "unified_headline_score",
]
corr_labels = ["Citation", "Cognitive", "Argument", "PeerReview", "Headline"]

corr_matrix = df[corr_cols].corr()
corr_matrix.index = corr_labels
corr_matrix.columns = corr_labels

print("\nPearson Correlation Matrix:")
print(corr_matrix.round(3).to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="RdYlGn",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax
)
ax.set_title("Cross-Module Score Correlations", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

#### 9.1.7 Consolidated Metrics Summary

All quantitative results in one table for easy reference.

In [ ]:
print("=" * 70)
print("CONSOLIDATED METRICS SUMMARY")
print("=" * 70)

summary_rows = []

# Citation module
summary_rows.append({
    "Module": "Citation Worthiness",
    "Type": "Classification",
    "Best Model": best_citation_model,
    "Primary Metric": "F1 (weighted)",
    "Value": max(lr_df["f1"].mean(), scibert_df["f1"].mean()),
    "Secondary Metric": "Accuracy",
    "Sec. Value": max(lr_df["accuracy"].mean(), scibert_df["accuracy"].mean()),
})

# Cognitive load (average across 3 targets)
summary_rows.append({
    "Module": "Cognitive Load",
    "Type": "Regression",
    "Best Model": ", ".join(final_test_results_df["best_model"].values),
    "Primary Metric": "RMSE (avg)",
    "Value": final_test_results_df["test_rmse"].mean(),
    "Secondary Metric": "R² (avg)",
    "Sec. Value": final_test_results_df["test_r2"].mean(),
})

# Argument quality (rule-based)
summary_rows.append({
    "Module": "Argument Quality",
    "Type": "Rule-based",
    "Best Model": "N/A",
    "Primary Metric": "Mean Score",
    "Value": df["argument_quality_score_final"].mean(),
    "Secondary Metric": "Coverage",
    "Sec. Value": 1.0,
})

# Peer review (rule-based)
summary_rows.append({
    "Module": "Peer Review",
    "Type": "Rule-based",
    "Best Model": "N/A",
    "Primary Metric": "Mean Score",
    "Value": df["peer_review_score"].mean(),
    "Secondary Metric": "Coverage",
    "Sec. Value": 1.0,
})

# Headline accuracy (classification)
summary_rows.append({
    "Module": "Headline Accuracy",
    "Type": "Classification",
    "Best Model": "Logistic Regression",
    "Primary Metric": "F1 (macro)",
    "Value": f1_score(y_hl_test, test_pred, average="macro"),
    "Secondary Metric": "ROC-AUC",
    "Sec. Value": roc_auc_score(y_hl_test, test_proba),
})

# Headline accuracy (regression)
summary_rows.append({
    "Module": "Headline Severity",
    "Type": "Regression",
    "Best Model": best_reg_name,
    "Primary Metric": "RMSE",
    "Value": np.sqrt(mean_squared_error(y_hl_score_test, test_score_pred)),
    "Secondary Metric": "R²",
    "Sec. Value": r2_score(y_hl_score_test, test_score_pred),
})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

### 9.2 Qualitative Evaluation

We examine generated reports at different quality levels to assess whether the system produces meaningful, coherent, and actionable feedback.

#### 9.2.1 Sample Reports at Different Quality Levels

We select papers at three quality tiers — high, medium, and low unified scores — and present their full reports to evaluate the system's diagnostic ability.

In [ ]:
# Identify papers at each tier
high_score_idx = df["unified_quality_score"].idxmax()
low_score_idx = df["unified_quality_score"].idxmin()
median_score = df["unified_quality_score"].median()
mid_score_idx = (df["unified_quality_score"] - median_score).abs().idxmin()

print("=" * 70)
print("QUALITATIVE EVALUATION — High Quality Paper")
print("=" * 70)
print(f"Unified Score: {df.loc[high_score_idx, 'unified_quality_score']:.1f}/100")
print(f"Level: {df.loc[high_score_idx, 'unified_quality_level']}")
print()
print(generate_unified_report(df.loc[high_score_idx]))

In [ ]:
print("=" * 70)
print("QUALITATIVE EVALUATION — Medium Quality Paper")
print("=" * 70)
print(f"Unified Score: {df.loc[mid_score_idx, 'unified_quality_score']:.1f}/100")
print(f"Level: {df.loc[mid_score_idx, 'unified_quality_level']}")
print()
print(generate_unified_report(df.loc[mid_score_idx]))

In [ ]:
print("=" * 70)
print("QUALITATIVE EVALUATION — Low Quality Paper")
print("=" * 70)
print(f"Unified Score: {df.loc[low_score_idx, 'unified_quality_score']:.1f}/100")
print(f"Level: {df.loc[low_score_idx, 'unified_quality_level']}")
print()
print(generate_unified_report(df.loc[low_score_idx]))

#### 9.2.2 Cross-Module Agreement Analysis

Do the modules agree on what constitutes a good or bad paper? We check how often papers that score high on one dimension also score high on others.

In [ ]:
print("=" * 70)
print("Cross-Module Agreement Analysis")
print("=" * 70)

# Define 'good' as above-median for each module
module_scores = {
    "Citation":  "unified_citation_score",
    "Cognitive":  "unified_cognitive_score",
    "Argument":   "unified_argument_score",
    "PeerReview": "unified_peer_review_score",
    "Headline":   "unified_headline_score",
}

# Count how many modules rate each paper as 'above median'
above_median = pd.DataFrame()
for name, col in module_scores.items():
    above_median[name] = (df[col] >= df[col].median()).astype(int)

df["modules_above_median"] = above_median.sum(axis=1)

print("\nNumber of modules rating paper above median:")
agreement_dist = df["modules_above_median"].value_counts().sort_index()
for count, n_papers in agreement_dist.items():
    pct = n_papers / len(df) * 100
    bar = "█" * int(pct / 2)
    label = "full agreement (all high)" if count == 5 else "full agreement (all low)" if count == 0 else "mixed"
    print(f"  {count}/5 modules above median: {n_papers:4d} papers ({pct:5.1f}%)  {bar}")

# Full agreement vs disagreement
full_agree = ((df["modules_above_median"] == 5) | (df["modules_above_median"] == 0)).sum()
full_agree_pct = full_agree / len(df) * 100
print(f"\nFull agreement (all 5 modules agree): {full_agree} papers ({full_agree_pct:.1f}%)")
print(f"Partial/mixed signals:                {len(df) - full_agree} papers ({100 - full_agree_pct:.1f}%)")

#### 9.2.3 Diagnostic Usefulness — Strengths & Weaknesses Coverage

A good quality assessment system should flag specific issues, not just assign a score. We check how often the rule-based modules identify concrete strengths, weaknesses, and missing elements.

In [ ]:
print("=" * 70)
print("Diagnostic Coverage Analysis")
print("=" * 70)

# Argument quality module
arg_has_issues = (df["argument_issues"].str.len() > 0).sum()
arg_has_strengths = (df["argument_strengths_text"].str.len() > 0).sum()

print("\nArgument Quality Module:")
print(f"  Papers with identified issues:    {arg_has_issues:4d} / {len(df)} ({arg_has_issues/len(df)*100:.1f}%)")
print(f"  Papers with identified strengths: {arg_has_strengths:4d} / {len(df)} ({arg_has_strengths/len(df)*100:.1f}%)")

# Peer review module
pr_has_weaknesses = (df["peer_review_weaknesses"].str.len() > 0).sum()
pr_has_strengths = (df["peer_review_strengths"].str.len() > 0).sum()
pr_has_missing = (df["peer_review_missing_elements"].str.len() > 0).sum()

print("\nPeer Review Module:")
print(f"  Papers with identified strengths:         {pr_has_strengths:4d} / {len(df)} ({pr_has_strengths/len(df)*100:.1f}%)")
print(f"  Papers with identified weaknesses:        {pr_has_weaknesses:4d} / {len(df)} ({pr_has_weaknesses/len(df)*100:.1f}%)")
print(f"  Papers with identified missing elements:  {pr_has_missing:4d} / {len(df)} ({pr_has_missing/len(df)*100:.1f}%)")

# Average number of diagnostic items per paper
avg_issues = df["argument_issues"].apply(lambda x: len(x.split(" | ")) if x else 0).mean()
avg_weaknesses = df["peer_review_weaknesses"].apply(lambda x: len(x.split(" | ")) if x else 0).mean()
avg_missing = df["peer_review_missing_elements"].apply(lambda x: len(x.split(" | ")) if x else 0).mean()

print(f"\nAverage diagnostic items per paper:")
print(f"  Argument issues:            {avg_issues:.1f}")
print(f"  Peer review weaknesses:     {avg_weaknesses:.1f}")
print(f"  Peer review missing items:  {avg_missing:.1f}")

#### 9.2.4 Case Study — Disagreement Analysis

When modules disagree (one scores high, another scores low), it often reveals meaningful nuance. We examine papers where the largest gap exists between the highest and lowest module scores.

In [ ]:
print("=" * 70)
print("Case Study: High Module Disagreement")
print("=" * 70)

# Compute per-paper score range (max module - min module)
score_cols = [
    "unified_citation_score",
    "unified_cognitive_score",
    "unified_argument_score",
    "unified_peer_review_score",
    "unified_headline_score",
]
df["module_score_range"] = df[score_cols].max(axis=1) - df[score_cols].min(axis=1)

# Find the paper with the highest disagreement
disagree_idx = df["module_score_range"].idxmax()
row = df.loc[disagree_idx]

print(f"\nPaper with largest module disagreement:")
print(f"  Title: {row['title']}")
print(f"  Score range: {row['module_score_range']:.1f} points")
print(f"\n  Module Scores:")
labels = ["Citation", "Cognitive", "Argument", "PeerReview", "Headline"]
for label, col in zip(labels, score_cols):
    val = row[col]
    bar = "█" * int(val / 5) + "░" * (20 - int(val / 5))
    print(f"    {label:12s}  {bar}  {val:.1f}")

print(f"\n  Unified Score: {row['unified_quality_score']:.1f} ({row['unified_quality_level']})")

print("\nFull report:")
print(generate_unified_report(row))

#### 9.2.5 Score Distribution Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

plot_configs = [
    ("unified_citation_score", "Citation Support", "steelblue"),
    ("unified_cognitive_score", "Cognitive Load", "seagreen"),
    ("unified_argument_score", "Argument Quality", "coral"),
    ("unified_peer_review_score", "Peer Review", "mediumpurple"),
    ("unified_headline_score", "Headline Accuracy", "goldenrod"),
    ("unified_quality_score", "Unified Score", "crimson"),
]

for i, (col, title, color) in enumerate(plot_configs):
    axes[i].hist(df[col], bins=15, color=color, edgecolor="white", alpha=0.85)
    axes[i].axvline(df[col].mean(), color="black", linestyle="--", linewidth=1.2, label=f"Mean: {df[col].mean():.1f}")
    axes[i].set_title(title, fontsize=11, fontweight="bold")
    axes[i].set_xlabel("Score (0-100)")
    axes[i].set_ylabel("Count")
    axes[i].legend(fontsize=9)

plt.suptitle("Module Score Distributions", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()